# TP 04 — Evaluation Metrics

**Recommender Systems — SCIA — Session 04**

Today you will build an offline evaluation pipeline for recommenders: split the data by time, implement the metrics yourself, and use them to compare seven recommenders on **accuracy and beyond-accuracy** criteria.

**Required standards:** modular code, numpy-style docstring on every function, self-contained notebook.


In [ ]:
import io
import os
import zipfile
import urllib.request
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

RELEVANCE_THRESHOLD = 4.0   # a held-out rating >= 4.0 means "the user liked it"
K = 10                      # length of the recommendation lists we evaluate


## 1. Data

Same MovieLens `ml-latest-small` as Session 01.

In [ ]:
def download_movielens(dest_dir: str, url: str = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip") -> str:
    """Download and extract the MovieLens ml-latest-small archive.

    Parameters
    ----------
    dest_dir : str
        Directory in which the archive is extracted. Created if needed.
    url : str, optional
        URL of the MovieLens zip archive.

    Returns
    -------
    str
        Path to the extracted ``ml-latest-small`` folder. If it already
        exists, nothing is downloaded.
    """
    os.makedirs(dest_dir, exist_ok=True)
    extracted_path = os.path.join(dest_dir, "ml-latest-small")
    if os.path.isdir(extracted_path):
        return extracted_path
    with urllib.request.urlopen(url) as response:
        archive_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        archive.extractall(dest_dir)
    return extracted_path


def load_movielens(data_dir: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load the MovieLens ratings and movies tables.

    Parameters
    ----------
    data_dir : str
        Path to the ``ml-latest-small`` folder.

    Returns
    -------
    ratings : pandas.DataFrame
        Columns ``userId``, ``movieId``, ``rating``, ``timestamp``.
    movies : pandas.DataFrame
        Columns ``movieId``, ``title``, ``genres``.
    """
    ratings = pd.read_csv(os.path.join(data_dir, "ratings.csv"))
    movies = pd.read_csv(os.path.join(data_dir, "movies.csv"))
    return ratings, movies


data_dir = download_movielens(dest_dir="./data")
ratings, movies = load_movielens(data_dir)
print(f"{len(ratings):,} ratings, {ratings['userId'].nunique()} users, {ratings['movieId'].nunique():,} rated movies")


## 2. Exercise 1 — a temporal split

Implement `temporal_user_split` following its docstring.

In [ ]:
def temporal_user_split(ratings: pd.DataFrame, test_frac: float = 0.2) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split ratings into train and test sets, per user, by time.

    For every user, ratings are sorted by ``timestamp`` (ties keep their
    original order in the file). The most recent
    ``max(1, floor(test_frac * n_user_ratings))`` ratings of that user go to
    the test set; all earlier ones go to the training set.

    Parameters
    ----------
    ratings : pandas.DataFrame
        Columns ``userId``, ``movieId``, ``rating``, ``timestamp``.
    test_frac : float, optional
        Fraction of each user's ratings to hold out (default 0.2).

    Returns
    -------
    train : pandas.DataFrame
        Training ratings, same columns as ``ratings``, index reset.
    test : pandas.DataFrame
        Held-out ratings, same columns as ``ratings``, index reset.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


In [ ]:
def restrict_to_known_items(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    """Drop test ratings of items that never appear in the training set.

    None of our recommenders can recommend an item it has never seen, so such
    test rows could never be hit by any model.

    Parameters
    ----------
    train : pandas.DataFrame
        Training ratings.
    test : pandas.DataFrame
        Held-out ratings.

    Returns
    -------
    pandas.DataFrame
        The rows of ``test`` whose ``movieId`` occurs in ``train``, index reset.
    """
    known = set(train["movieId"])
    return test[test["movieId"].isin(known)].reset_index(drop=True)


In [ ]:
train, test_all = temporal_user_split(ratings, test_frac=0.2)
test = restrict_to_known_items(train, test_all)

print(f"train: {len(train):,} rows (expected 80,896)   test before filtering: {len(test_all):,} rows (expected 19,940)")
print(f"test after dropping unseen items: {len(test):,} rows")

# No leakage: for every user, nothing in train is more recent than anything in test.
last_train = train.groupby("userId")["timestamp"].max()
first_test = test_all.groupby("userId")["timestamp"].min()
assert (last_train <= first_test).all(), "Some user has a training rating newer than a test rating."
print("No temporal leakage.")


## 3. Exercise 2 — ranking metrics

Implement `precision_at_k`, `recall_at_k` and `ndcg_at_k` for one user's list.

In [ ]:
def precision_at_k(recommended: Sequence[int], relevant: set[int], k: int) -> float:
    """Precision@K of one recommendation list.

    ``|top-K recommended items that are relevant| / K``. The denominator is
    always ``k``, even if fewer than ``k`` items were recommended.

    Parameters
    ----------
    recommended : sequence of int
        Recommended item ids, best first.
    relevant : set of int
        Ids of the items that are relevant for this user.
    k : int
        List length to evaluate.

    Returns
    -------
    float
        Precision@K, in [0, 1].

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


def recall_at_k(recommended: Sequence[int], relevant: set[int], k: int) -> float:
    """Recall@K of one recommendation list.

    ``|top-K recommended items that are relevant| / |relevant|``.

    Parameters
    ----------
    recommended : sequence of int
        Recommended item ids, best first.
    relevant : set of int
        Ids of the items that are relevant for this user.
    k : int
        List length to evaluate.

    Returns
    -------
    float
        Recall@K, in [0, 1]. Returns 0.0 if ``relevant`` is empty.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


def ndcg_at_k(recommended: Sequence[int], relevant: set[int], k: int) -> float:
    """NDCG@K of one recommendation list, with binary relevance.

    ``DCG@K = sum_i rel_i / log2(i + 1)`` with ranks ``i`` starting at 1 and
    ``rel_i`` equal to 1 if the item at rank ``i`` is relevant, else 0.
    ``IDCG@K`` is the DCG@K of the best possible list given ``relevant``,
    i.e. one with ``min(|relevant|, K)`` relevant items in the first
    positions. ``NDCG@K = DCG@K / IDCG@K``.

    Parameters
    ----------
    recommended : sequence of int
        Recommended item ids, best first.
    relevant : set of int
        Ids of the items that are relevant for this user.
    k : int
        List length to evaluate.

    Returns
    -------
    float
        NDCG@K, in [0, 1]. Returns 0.0 if ``relevant`` is empty.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


In [ ]:
recommended = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
relevant = {10, 30, 40, 70, 200}

for k, (p_exp, r_exp, n_exp) in {5: (0.60, 0.60, 0.6548), 10: (0.40, 0.80, 0.7679)}.items():
    print(f"K={k:2d}  P={precision_at_k(recommended, relevant, k):.2f} (expected {p_exp:.2f})   "
          f"R={recall_at_k(recommended, relevant, k):.2f} (expected {r_exp:.2f})   "
          f"NDCG={ndcg_at_k(recommended, relevant, k):.4f} (expected {n_exp:.4f})")


## 4. Exercise 3 — rating-prediction error

Implement `rmse` and `mae`, then compare two simple rating predictors on the test set.

In [ ]:
def rmse(y_true: Sequence[float], y_pred: Sequence[float]) -> float:
    """Root mean squared error.

    Parameters
    ----------
    y_true : sequence of float
        True ratings.
    y_pred : sequence of float
        Predicted ratings, same length as ``y_true``.

    Returns
    -------
    float
        ``sqrt(mean((y_true - y_pred) ** 2))``.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


def mae(y_true: Sequence[float], y_pred: Sequence[float]) -> float:
    """Mean absolute error.

    Parameters
    ----------
    y_true : sequence of float
        True ratings.
    y_pred : sequence of float
        Predicted ratings, same length as ``y_true``.

    Returns
    -------
    float
        ``mean(abs(y_true - y_pred))``.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


In [ ]:
def predict_global_mean(train: pd.DataFrame, test: pd.DataFrame) -> np.ndarray:
    """Predict every test rating as the global mean training rating.

    Parameters
    ----------
    train : pandas.DataFrame
        Training ratings.
    test : pandas.DataFrame
        Held-out ratings.

    Returns
    -------
    numpy.ndarray, shape (len(test),)
        The same value repeated for every test row.
    """
    return np.full(len(test), train["rating"].mean())


def predict_item_mean(train: pd.DataFrame, test: pd.DataFrame) -> np.ndarray:
    """Predict each test rating as the item's mean training rating.

    Parameters
    ----------
    train : pandas.DataFrame
        Training ratings.
    test : pandas.DataFrame
        Held-out ratings. Every item is known from ``train`` (see
        ``restrict_to_known_items``).

    Returns
    -------
    numpy.ndarray, shape (len(test),)
        Per-row predictions, aligned with the rows of ``test``.
    """
    item_means = train.groupby("movieId")["rating"].mean()
    return test["movieId"].map(item_means).fillna(train["rating"].mean()).to_numpy()


In [ ]:
assert np.isclose(rmse([4, 3, 5, 2, 1], [5, 3, 4, 2, 1]), 0.6325, atol=1e-4)
assert np.isclose(mae([4, 3, 5, 2, 1], [5, 3, 4, 2, 1]), 0.40)

for name, predictor in {"global mean": predict_global_mean, "item mean": predict_item_mean}.items():
    pred = predictor(train, test)
    print(f"{name:12s} RMSE = {rmse(test['rating'], pred):.4f}   MAE = {mae(test['rating'], pred):.4f}")


## 5. Exercise 4 — beyond accuracy

Implement `catalog_coverage`, `intra_list_diversity` and `novelty`.

In [ ]:
def catalog_coverage(rec_lists: Iterable[Sequence[int]], n_catalogue: int) -> float:
    """Catalogue coverage of a set of recommendation lists.

    Parameters
    ----------
    rec_lists : iterable of sequence of int
        One recommendation list per user.
    n_catalogue : int
        Number of items in the catalogue.

    Returns
    -------
    float
        Number of distinct items appearing in at least one list, divided by
        ``n_catalogue``.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


def intra_list_diversity(recommended: Sequence[int], item_vectors, item_index: dict[int, int]) -> float:
    """Intra-list diversity of one recommendation list.

    Parameters
    ----------
    recommended : sequence of int
        Recommended item ids.
    item_vectors : scipy.sparse matrix or numpy.ndarray, shape (n_items, n_features)
        One feature vector per row.
    item_index : dict of {int: int}
        Maps an item id to its row in ``item_vectors``. Items missing from
        this mapping are ignored.

    Returns
    -------
    float
        Mean, over all pairs of items in the list, of
        ``1 - cosine_similarity``. Returns 0.0 if fewer than two items are
        known.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


def novelty(recommended: Sequence[int], item_counts: dict[int, int], n_users: int) -> float:
    """Novelty of one recommendation list, in bits.

    Parameters
    ----------
    recommended : sequence of int
        Recommended item ids.
    item_counts : dict of {int: int}
        Number of training users who rated each item. Items missing from the
        dictionary, or with a count below 1, are treated as having count 1.
    n_users : int
        Number of users in the training set.

    Returns
    -------
    float
        Mean over the list of ``-log2(count_i / n_users)``.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


In [ ]:
toy_vectors = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
toy_index = {10: 0, 20: 1, 30: 2}

print(f"coverage          {catalog_coverage([[1, 2], [2, 3]], n_catalogue=10):.2f}   (expected 0.30)")
print(f"diversity [10,20] {intra_list_diversity([10, 20], toy_vectors, toy_index):.4f} (expected 1.0000)")
print(f"diversity [10,30] {intra_list_diversity([10, 30], toy_vectors, toy_index):.4f} (expected 0.2929)")
print(f"novelty           {novelty([10, 20], {10: 50, 20: 5}, n_users=100):.4f} (expected 2.6610)")
print(f"novelty (unknown) {novelty([30], {}, n_users=100):.4f} (expected 6.6439)")


## 6. The recommenders to evaluate

These are provided: they are the systems under test, not the exercise. They reuse ideas from Sessions 01 and 02 (popularity, genre content-based). Read them, but you do not need to modify them.


In [ ]:
def build_genre_vectors(movies: pd.DataFrame, items: np.ndarray):
    """TF-IDF vectors of the genres of each item.

    Parameters
    ----------
    movies : pandas.DataFrame
        Columns ``movieId`` and ``genres`` (pipe-separated).
    items : numpy.ndarray
        Item ids to vectorise, in the row order of the result.

    Returns
    -------
    vectors : scipy.sparse.csr_matrix, shape (len(items), n_genres)
        One TF-IDF row per item.
    index : dict of {int: int}
        Maps an item id to its row in ``vectors``.
    """
    genres = movies.set_index("movieId").loc[items, "genres"].replace("(no genres listed)", "")
    vectoriser = TfidfVectorizer(tokenizer=lambda text: [g for g in text.split("|") if g],
                                 token_pattern=None, lowercase=False)
    return vectoriser.fit_transform(genres), {int(m): i for i, m in enumerate(items)}


class RandomRecommender:
    """Recommend unseen items uniformly at random.

    Parameters
    ----------
    seed : int, optional
        Seed of the random number generator.

    Attributes
    ----------
    items_ : numpy.ndarray
        Catalogue (items seen in the training set).
    seen_ : dict of {int: set of int}
        Items each user rated in training.
    """

    def __init__(self, seed: int = 0):
        self.seed = seed

    def fit(self, train: pd.DataFrame, movies: pd.DataFrame) -> "RandomRecommender":
        """Store the catalogue and each user's seen items.

        Parameters
        ----------
        train : pandas.DataFrame
            Training ratings.
        movies : pandas.DataFrame
            Movie metadata (unused here).

        Returns
        -------
        RandomRecommender
            The fitted recommender.
        """
        self.items_ = np.array(sorted(train["movieId"].unique()))
        self.seen_ = train.groupby("userId")["movieId"].apply(set).to_dict()
        self.rng_ = np.random.default_rng(self.seed)
        return self

    def recommend(self, user_id: int, k: int) -> list[int]:
        """Return ``k`` random items the user has not rated.

        Parameters
        ----------
        user_id : int
            User id.
        k : int
            List length.

        Returns
        -------
        list of int
            Item ids.
        """
        pool = self.items_[~np.isin(self.items_, list(self.seen_.get(user_id, set())))]
        return self.rng_.choice(pool, size=k, replace=False).tolist()


class MostPopularRecommender:
    """Recommend the items with the most training ratings that the user has not rated.

    Attributes
    ----------
    ranking_ : list of int
        Items sorted by number of ratings, descending.
    seen_ : dict of {int: set of int}
        Items each user rated in training.
    """

    def fit(self, train: pd.DataFrame, movies: pd.DataFrame) -> "MostPopularRecommender":
        """Rank items by number of ratings.

        Parameters
        ----------
        train : pandas.DataFrame
            Training ratings.
        movies : pandas.DataFrame
            Movie metadata (unused here).

        Returns
        -------
        MostPopularRecommender
            The fitted recommender.
        """
        self.ranking_ = train.groupby("movieId").size().sort_values(ascending=False, kind="mergesort").index.tolist()
        self.seen_ = train.groupby("userId")["movieId"].apply(set).to_dict()
        return self

    def recommend(self, user_id: int, k: int) -> list[int]:
        """Return the ``k`` most-rated items the user has not rated.

        Parameters
        ----------
        user_id : int
            User id.
        k : int
            List length.

        Returns
        -------
        list of int
            Item ids.
        """
        seen = self.seen_.get(user_id, set())
        return [item for item in self.ranking_ if item not in seen][:k]


class BestRatedRecommender:
    """Recommend the highest mean-rated items among those with enough ratings (the Session 01 baseline).

    Parameters
    ----------
    min_ratings : int, optional
        Minimum number of training ratings for an item to be eligible.

    Attributes
    ----------
    ranking_ : list of int
        Eligible items sorted by mean rating, descending.
    seen_ : dict of {int: set of int}
        Items each user rated in training.
    """

    def __init__(self, min_ratings: int = 50):
        self.min_ratings = min_ratings

    def fit(self, train: pd.DataFrame, movies: pd.DataFrame) -> "BestRatedRecommender":
        """Rank eligible items by mean rating.

        Parameters
        ----------
        train : pandas.DataFrame
            Training ratings.
        movies : pandas.DataFrame
            Movie metadata (unused here).

        Returns
        -------
        BestRatedRecommender
            The fitted recommender.
        """
        stats = train.groupby("movieId")["rating"].agg(["mean", "size"])
        stats = stats[stats["size"] >= self.min_ratings].sort_values("mean", ascending=False, kind="mergesort")
        self.ranking_ = stats.index.tolist()
        self.seen_ = train.groupby("userId")["movieId"].apply(set).to_dict()
        return self

    def recommend(self, user_id: int, k: int) -> list[int]:
        """Return the ``k`` best-rated eligible items the user has not rated.

        Parameters
        ----------
        user_id : int
            User id.
        k : int
            List length.

        Returns
        -------
        list of int
            Item ids.
        """
        seen = self.seen_.get(user_id, set())
        return [item for item in self.ranking_ if item not in seen][:k]


class GenreContentRecommender:
    """Content-based recommender on genre TF-IDF vectors (Session 02 idea).

    A user profile is the mean vector of the items the user liked in
    training; items are scored by cosine similarity to the profile.

    Parameters
    ----------
    threshold : float, optional
        Minimum training rating for an item to count as liked.
    pop_weight : float, optional
        Weight of a popularity prior added to the cosine score. With genres
        only, many items have identical vectors; a small weight breaks those
        ties by popularity, a larger one blends in popularity (a first taste
        of the hybrids of Session 05).

    Attributes
    ----------
    items_ : numpy.ndarray
        Catalogue (items seen in training).
    vectors_ : scipy.sparse.csr_matrix
        Genre TF-IDF vectors of ``items_``.
    index_ : dict of {int: int}
        Item id to row of ``vectors_``.
    pop_ : numpy.ndarray
        Popularity of each item in [0, 1] (rating count / max rating count).
    seen_ : dict of {int: set of int}
        Items each user rated in training.
    liked_ : dict of {int: list of int}
        Items each user liked in training.
    """

    def __init__(self, threshold: float = RELEVANCE_THRESHOLD, pop_weight: float = 0.0):
        self.threshold = threshold
        self.pop_weight = pop_weight

    def fit(self, train: pd.DataFrame, movies: pd.DataFrame) -> "GenreContentRecommender":
        """Build item vectors, popularity and per-user histories.

        Parameters
        ----------
        train : pandas.DataFrame
            Training ratings.
        movies : pandas.DataFrame
            Movie metadata with a ``genres`` column.

        Returns
        -------
        GenreContentRecommender
            The fitted recommender.
        """
        self.items_ = np.array(sorted(train["movieId"].unique()))
        self.vectors_, self.index_ = build_genre_vectors(movies, self.items_)
        counts = train.groupby("movieId").size().reindex(self.items_).to_numpy(float)
        self.pop_ = counts / counts.max()
        self.seen_ = train.groupby("userId")["movieId"].apply(set).to_dict()
        liked = train[train["rating"] >= self.threshold]
        self.liked_ = liked.groupby("userId")["movieId"].apply(list).to_dict()
        return self

    def scores(self, user_id: int) -> np.ndarray:
        """Score every catalogue item for one user.

        Parameters
        ----------
        user_id : int
            User id.

        Returns
        -------
        numpy.ndarray, shape (len(items_),)
            Cosine similarity to the user profile plus ``pop_weight * pop_``.
        """
        liked = self.liked_.get(user_id) or list(self.seen_.get(user_id, []))
        rows = [self.index_[item] for item in liked if item in self.index_]
        profile = np.asarray(self.vectors_[rows].mean(axis=0))
        return cosine_similarity(profile, self.vectors_).ravel() + self.pop_weight * self.pop_

    def recommend(self, user_id: int, k: int) -> list[int]:
        """Return the ``k`` best-scoring items the user has not rated.

        Parameters
        ----------
        user_id : int
            User id.
        k : int
            List length.

        Returns
        -------
        list of int
            Item ids.
        """
        seen = self.seen_.get(user_id, set())
        out = []
        for j in np.argsort(-self.scores(user_id), kind="mergesort"):
            item = int(self.items_[j])
            if item not in seen:
                out.append(item)
                if len(out) == k:
                    break
        return out


class MMRReranker:
    """Re-rank a content-based recommender with Maximal Marginal Relevance.

    Items are picked greedily: each new item maximises
    ``lambda_ * relevance - (1 - lambda_) * (max similarity to items already picked)``.

    Parameters
    ----------
    base : GenreContentRecommender
        The recommender whose candidates are re-ranked.
    lambda_ : float, optional
        Trade-off in [0, 1]; 1 keeps the base ranking, lower values favour diversity.
    n_candidates : int, optional
        Number of top base candidates to re-rank.
    """

    def __init__(self, base: GenreContentRecommender, lambda_: float = 0.7, n_candidates: int = 100):
        self.base = base
        self.lambda_ = lambda_
        self.n_candidates = n_candidates

    def fit(self, train: pd.DataFrame, movies: pd.DataFrame) -> "MMRReranker":
        """Fit the base recommender.

        Parameters
        ----------
        train : pandas.DataFrame
            Training ratings.
        movies : pandas.DataFrame
            Movie metadata.

        Returns
        -------
        MMRReranker
            The fitted re-ranker.
        """
        self.base.fit(train, movies)
        return self

    def recommend(self, user_id: int, k: int) -> list[int]:
        """Return ``k`` items selected by MMR.

        Parameters
        ----------
        user_id : int
            User id.
        k : int
            List length.

        Returns
        -------
        list of int
            Item ids.
        """
        scores = self.base.scores(user_id)
        seen = self.base.seen_.get(user_id, set())
        order = [j for j in np.argsort(-scores, kind="mergesort") if int(self.base.items_[j]) not in seen]
        cand = np.array(order[: self.n_candidates])
        relevance = scores[cand]
        similarity = cosine_similarity(self.base.vectors_[cand])
        chosen, remaining = [], list(range(len(cand)))
        while len(chosen) < k and remaining:
            if not chosen:
                best = remaining[int(np.argmax(relevance[remaining]))]
            else:
                redundancy = similarity[np.ix_(remaining, chosen)].max(axis=1)
                best = remaining[int(np.argmax(self.lambda_ * relevance[remaining] - (1 - self.lambda_) * redundancy))]
            chosen.append(best)
            remaining.remove(best)
        return [int(self.base.items_[cand[c]]) for c in chosen]


## 7. Exercise 5 — the evaluation loop

Implement `evaluate_recommender`: for every evaluable user, build the list and compute the five per-user metrics.

In [ ]:
def evaluate_recommender(
    recommender,
    test: pd.DataFrame,
    k: int,
    item_counts: dict[int, int],
    n_users: int,
    item_vectors,
    item_index: dict[int, int],
    threshold: float = RELEVANCE_THRESHOLD,
) -> tuple[pd.DataFrame, dict[int, list[int]]]:
    """Evaluate one recommender on a held-out test set.

    A user is evaluated only if they have at least one relevant held-out
    item (a test rating ``>= threshold``). For each evaluated user, the
    recommender produces a top-``k`` list with ``recommender.recommend(user_id, k)``
    and the following per-user metrics are computed: precision@k, recall@k,
    ndcg@k, intra-list diversity and novelty.

    Parameters
    ----------
    recommender : object
        Fitted recommender exposing ``recommend(user_id, k) -> list[int]``.
    test : pandas.DataFrame
        Held-out ratings (``userId``, ``movieId``, ``rating``, ...).
    k : int
        List length.
    item_counts : dict of {int: int}
        Training rating counts per item (used by ``novelty``).
    n_users : int
        Number of training users (used by ``novelty``).
    item_vectors : scipy.sparse matrix or numpy.ndarray
        Item feature vectors (used by ``intra_list_diversity``).
    item_index : dict of {int: int}
        Item id to row of ``item_vectors``.
    threshold : float, optional
        Minimum test rating for an item to count as relevant.

    Returns
    -------
    per_user : pandas.DataFrame
        Indexed by ``userId``; columns ``precision``, ``recall``, ``ndcg``,
        ``diversity``, ``novelty``; one row per evaluated user.
    rec_lists : dict of {int: list of int}
        The recommended list of every evaluated user.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


In [ ]:
def summarise_results(per_user: pd.DataFrame, rec_lists: dict[int, list[int]], n_catalogue: int) -> dict:
    """Aggregate per-user metrics into one row of the results table.

    Parameters
    ----------
    per_user : pandas.DataFrame
        Output of ``evaluate_recommender``.
    rec_lists : dict of {int: list of int}
        Output of ``evaluate_recommender``.
    n_catalogue : int
        Number of items in the catalogue.

    Returns
    -------
    dict
        Mean ``precision``, ``recall``, ``ndcg``, ``diversity``, ``novelty``;
        ``precision_se`` (standard error of the mean precision over users);
        ``coverage``; and ``n_users``.
    """
    n = len(per_user)
    return {
        "precision": per_user["precision"].mean(),
        "precision_se": per_user["precision"].std(ddof=1) / np.sqrt(n),
        "recall": per_user["recall"].mean(),
        "ndcg": per_user["ndcg"].mean(),
        "coverage": catalog_coverage(rec_lists.values(), n_catalogue),
        "diversity": per_user["diversity"].mean(),
        "novelty": per_user["novelty"].mean(),
        "n_users": n,
    }


## 8. Run the comparison

Fit all seven recommenders on the training set and evaluate them on the held-out ratings (this takes about a minute).

In [ ]:
items = np.array(sorted(train["movieId"].unique()))
item_vectors, item_index = build_genre_vectors(movies, items)
item_counts = train.groupby("movieId").size().to_dict()
n_train_users = train["userId"].nunique()

models = {
    "Random": RandomRecommender(seed=0),
    "Best-rated (min 1)": BestRatedRecommender(min_ratings=1),
    "Best-rated (min 50) [S01]": BestRatedRecommender(min_ratings=50),
    "Most popular": MostPopularRecommender(),
    "Content (genres)": GenreContentRecommender(pop_weight=0.001),
    "Content + popularity prior": GenreContentRecommender(pop_weight=0.3),
    "Content + prior + MMR 0.5": MMRReranker(GenreContentRecommender(pop_weight=0.3), lambda_=0.5),
}

rows = {}
for name, model in models.items():
    model.fit(train, movies)
    per_user, rec_lists = evaluate_recommender(model, test, K, item_counts, n_train_users, item_vectors, item_index)
    rows[name] = summarise_results(per_user, rec_lists, n_catalogue=len(items))

results = pd.DataFrame(rows).T
results.round(4)


In [ ]:
def plot_results(results: pd.DataFrame) -> None:
    """Plot precision (with standard error) and the precision/diversity trade-off.

    Parameters
    ----------
    results : pandas.DataFrame
        Indexed by model name; columns ``precision``, ``precision_se``, ``diversity``.
    """
    fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4.8))
    left.barh(results.index, results["precision"], xerr=1.96 * results["precision_se"], color="#56c8ba", capsize=3)
    left.invert_yaxis()
    left.set_xlabel(f"Precision@{K} (bars: 95% interval)")
    right.scatter(results["diversity"], results["precision"], s=70, color="#e8c47a")
    for name, row in results.iterrows():
        right.annotate(name, (row["diversity"], row["precision"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
    right.set_xlabel("Intra-list diversity")
    right.set_ylabel(f"Precision@{K}")
    plt.tight_layout()
    plt.show()


def log_results(comparison_path: str, results: pd.DataFrame, session: str = "S04") -> pd.DataFrame:
    """Append one row per model to the running comparison table.

    Parameters
    ----------
    comparison_path : str
        CSV path of the running comparison table (created if missing).
    results : pandas.DataFrame
        Indexed by model name, one row per model.
    session : str, optional
        Session label written in a ``session`` column.

    Returns
    -------
    pandas.DataFrame
        The updated comparison table.
    """
    new_rows = results.reset_index().rename(columns={"index": "model"})
    new_rows.insert(0, "session", session)
    if os.path.exists(comparison_path):
        new_rows = pd.concat([pd.read_csv(comparison_path), new_rows], ignore_index=True)
    new_rows.to_csv(comparison_path, index=False)
    return new_rows


In [ ]:
plot_results(results)

## 9. Log to the comparison table

In [ ]:
table = log_results("./comparison_table.csv", results)
table.tail(len(results))


## 10. Going further (open-ended)

Pick at least one, and **justify your conclusion with the standard errors**.

**1. The MMR trade-off.** Re-run the last model for several values of `lambda_` (for example 0.9, 0.7, 0.5, 0.3, 0.1) and plot precision against diversity. Is there a trade-off? Is any difference larger than the noise?

**2. Serendipity.** Implement `serendipity_at_k(recommended, relevant, expected, k)` as the number of relevant recommended items that are *not* in `expected` (for instance the 100 most popular items), divided by `k`. Which model is the most serendipitous?

**3. Which model would you ship?** Choose one model for a "top picks for you" shelf and another for a "discover something new" page. Defend each choice with at least three of the metrics above.

**4. Your Session 03 model.** What would it take to evaluate the latent-factor model from Session 03 with this pipeline? (Think about data, split and the `recommend` method before writing code.)

---
**End of TP 04.**
